# Features, targets and leakage

How to build time-series features **that would actually have been available at prediction time**.

Every section starts with a tiny series of 6 hourly values (10, 20, 30, …) so you can see
exactly which value moves where. Only after that is the same thing done on the real data.

**What's in here**
- writing the prediction problem down before touching code
- the target: `shift(-h)`, and the separate-`dropna` trap
- lag features: `shift(k)`
- rolling features and why `shift(1)` comes before `rolling`
- `diff` and `pct_change`
- calendar features: numeric, one-hot, sin/cos
- exogenous variables: the forecast that existed at decision time (`merge_asof`)
- scaler leakage, group-mean leakage
- overlapping labels and shuffled cross-validation
- `bfill` / `interpolate` leakage
- `resample` label/closed
- a leakage checklist

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 0. Write the problem down first

> **At time t, using only information available at t, predict y at t + h.**

Fix three things before writing any feature:
1. **Decision time** t: when the forecast is made.
2. **Horizon** h: the target is at t + h.
3. **Information set**: which numbers, with which timestamps, were known at t.

**Interview check:** "What is one row, and at what moment would you have had these numbers?"

The toy series used everywhere below: six hours, values 10 to 60.

In [2]:
idx = pd.date_range("2023-01-01 00:00", periods=6, freq="h")
y = pd.Series([10, 20, 30, 40, 50, 60], index=idx, name="y")
y

2023-01-01 00:00:00    10
2023-01-01 01:00:00    20
2023-01-01 02:00:00    30
2023-01-01 03:00:00    40
2023-01-01 04:00:00    50
2023-01-01 05:00:00    60
Freq: h, Name: y, dtype: int64

## 1. The target: `shift(-h)`

`shift(-h)` moves values **up** by h rows, so the row at time t holds the value from t + h.
The last h rows have no future yet and become NaN.

In [3]:
pd.DataFrame({"y": y, "target_h1": y.shift(-1), "target_h2": y.shift(-2)})

,y,target_h1,target_h2
2023-01-01 00:00:00,10,20.0,30.0
2023-01-01 01:00:00,20,30.0,40.0
2023-01-01 02:00:00,30,40.0,50.0
2023-01-01 03:00:00,40,50.0,60.0
2023-01-01 04:00:00,50,60.0,NaN
2023-01-01 05:00:00,60,NaN,NaN


At 00:00 the h=1 target is 20 (the value at 01:00) and the h=2 target is 30 (the value at 02:00).
The last one (h=1) and two (h=2) rows are NaN.

**Pitfall: `dropna` on X and y separately.** A lag feature loses rows at the **start**,
the target loses rows at the **end**. Drop them separately and the two no longer line up.

In [4]:
X = pd.DataFrame({"lag1": y.shift(1)})
target = y.shift(-2)
print(X)
print()
print(target)

                     lag1
2023-01-01 00:00:00   NaN
2023-01-01 01:00:00  10.0
2023-01-01 02:00:00  20.0
2023-01-01 03:00:00  30.0
2023-01-01 04:00:00  40.0
2023-01-01 05:00:00  50.0

2023-01-01 00:00:00    30.0
2023-01-01 01:00:00    40.0
2023-01-01 02:00:00    50.0
2023-01-01 03:00:00    60.0
2023-01-01 04:00:00     NaN
2023-01-01 05:00:00     NaN
Freq: h, Name: y, dtype: float64


In [5]:
X_bad = X.dropna()
target_bad = target.dropna()
print("X rows      :", list(X_bad.index.strftime("%H:%M")))
print("target rows :", list(target_bad.index.strftime("%H:%M")))

X rows      : ['01:00', '02:00', '03:00', '04:00', '05:00']
target rows : ['00:00', '01:00', '02:00', '03:00']


X has 01:00–05:00, the target has 00:00–03:00. Same length (5 vs 4, or sometimes equal by
accident), different hours. Passing `.values` of both to a model would pair 01:00's feature
with 00:00's target.

**Correct pattern:** put target and features in **one** frame, `dropna` once, then split.

In [6]:
frame = X.copy()
frame["target"] = target
frame

,lag1,target
2023-01-01 00:00:00,NaN,30.0
2023-01-01 01:00:00,10.0,40.0
2023-01-01 02:00:00,20.0,50.0
2023-01-01 03:00:00,30.0,60.0
2023-01-01 04:00:00,40.0,NaN
2023-01-01 05:00:00,50.0,NaN


In [7]:
frame = frame.dropna()
frame

,lag1,target
2023-01-01 01:00:00,10.0,40.0
2023-01-01 02:00:00,20.0,50.0
2023-01-01 03:00:00,30.0,60.0


Only the rows where **both** exist survive (01:00–03:00), and feature and target sit on the same row.

On the real data: hourly consumption, target 24 hours ahead. Check one row by hand.

In [8]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
cons = df["consumption_mwh"]
tgt = pd.DataFrame({"time": df["time"], "y_now": cons, "y_h24": cons.shift(-24)})
tgt.head(3)

,time,y_now,y_h24
0,2022-01-01 00:00:00+00:00,26858.4,26785.3
1,2022-01-01 01:00:00+00:00,26177.8,25826.5
2,2022-01-01 02:00:00+00:00,26229.4,25799.8


In [9]:
print("target at row 0     :", tgt.loc[0, "y_h24"])
print("actual at row 24    :", tgt.loc[24, "y_now"])
print("NaN targets at end  :", tgt["y_h24"].isna().sum())

target at row 0     : 26785.3
actual at row 24    : 26785.3
NaN targets at end  : 24


## 2. Lag features: `shift(k)`

Positive k looks **back**: the row at t gets the value from t − k. The first k rows have no past and become NaN.

In [10]:
pd.DataFrame({"y": y, "lag1": y.shift(1), "lag2": y.shift(2)})

,y,lag1,lag2
2023-01-01 00:00:00,10,NaN,NaN
2023-01-01 01:00:00,20,10.0,NaN
2023-01-01 02:00:00,30,20.0,10.0
2023-01-01 03:00:00,40,30.0,20.0
2023-01-01 04:00:00,50,40.0,30.0
2023-01-01 05:00:00,60,50.0,40.0


At 02:00, `lag1` is 20 (the 01:00 value) and `lag2` is 10 (the 00:00 value).

**Pitfall:** `shift` counts **rows**, not hours. If an hour is missing, `shift(1)` is no longer "one hour ago".
Remove 02:00 from the toy series and look at 03:00.

In [11]:
y_gap = y.drop(pd.Timestamp("2023-01-01 02:00"))
pd.DataFrame({"y": y_gap, "lag1_by_rows": y_gap.shift(1), "lag1_by_time": y_gap.shift(1, freq="h")})

,y,lag1_by_rows,lag1_by_time
2023-01-01 00:00:00,10.0,NaN,NaN
2023-01-01 01:00:00,20.0,10.0,10.0
2023-01-01 02:00:00,NaN,NaN,20.0
2023-01-01 03:00:00,40.0,20.0,NaN
2023-01-01 04:00:00,50.0,40.0,40.0
2023-01-01 05:00:00,60.0,50.0,50.0
2023-01-01 06:00:00,NaN,NaN,60.0


At 03:00, `lag1_by_rows` is 20 — the value from 01:00, two hours earlier. `shift(1, freq="h")`
moves the *timestamps* instead and gives the honest answer: nothing was observed at 02:00, so
NaN at 03:00. (The table shows 02:00 and 06:00 as rows with `y` = NaN only because the shifted
column carries those labels; they are not in `y_gap`.)

Before using `shift(k)` on real data, check the interval is constant:

In [12]:
df["time"].diff().value_counts()

time
0 days 01:00:00    17519
Name: count, dtype: int64

One value only (1 hour): the grid is complete, so `shift(24)` really is 24 hours.

## 3. Rolling features: `shift(1)` before `rolling`

`rolling(2).mean()` at row t averages the values at t−1 **and t**. It includes the current row.

In [13]:
pd.DataFrame({"y": y, "rolling2": y.rolling(2).mean()})

,y,rolling2
2023-01-01 00:00:00,10,NaN
2023-01-01 01:00:00,20,15.0
2023-01-01 02:00:00,30,25.0
2023-01-01 03:00:00,40,35.0
2023-01-01 04:00:00,50,45.0
2023-01-01 05:00:00,60,55.0


At 01:00 the rolling mean is 15 = (10 + 20) / 2: it used the 01:00 value itself.
If 01:00's value is the thing you predict, the feature contains the answer.

`shift(1)` first, then the window ends one row **before** t.

In [14]:
pd.DataFrame({
    "y": y,
    "rolling2 (includes y[t])": y.rolling(2).mean(),
    "shift(1).rolling2 (ends at t-1)": y.shift(1).rolling(2).mean(),
})

,y,rolling2 (includes y[t]),shift(1).rolling2 (ends at t-1)
2023-01-01 00:00:00,10,NaN,NaN
2023-01-01 01:00:00,20,15.0,NaN
2023-01-01 02:00:00,30,25.0,15.0
2023-01-01 03:00:00,40,35.0,25.0
2023-01-01 04:00:00,50,45.0,35.0
2023-01-01 05:00:00,60,55.0,45.0


At 02:00: the un-shifted window gives 25 = (20 + 30) / 2 and includes 30, the value at 02:00.
The shifted window gives 15 = (10 + 20) / 2 and uses only the past.

**Pitfall:** `center=True` uses future rows on both sides. Never a feature.

In [15]:
pd.DataFrame({"y": y, "rolling3 center=True": y.rolling(3, center=True).mean()})

,y,rolling3 center=True
2023-01-01 00:00:00,10,NaN
2023-01-01 01:00:00,20,20.0
2023-01-01 02:00:00,30,30.0
2023-01-01 03:00:00,40,40.0
2023-01-01 04:00:00,50,50.0
2023-01-01 05:00:00,60,NaN


At 01:00 the centred mean is 20 = (10 + 20 + 30) / 3 — it used the value at 02:00, one hour in the future.

**How big is the leak?** Predict consumption at t from calendar features plus one rolling feature.
First with `rolling(3)` (window includes the target), then with `shift(1).rolling(3)`.

In [16]:
feat = pd.DataFrame({
    "hour": df["time"].dt.hour,
    "dow": df["time"].dt.dayofweek,
    "roll3_leaky": cons.rolling(3).mean(),
    "roll3_honest": cons.shift(1).rolling(3).mean(),
    "y": cons,
}).dropna()
split = int(len(feat) * 0.8)
train, test = feat.iloc[:split], feat.iloc[split:]
feat.head(4)

,hour,dow,roll3_leaky,roll3_honest,y
3,3,5,25929.500000,26421.866667,25381.3
4,4,5,25611.233333,25929.500000,25223.0
5,5,5,25686.000000,25611.233333,26453.7
6,6,5,27043.700000,25686.000000,29454.4


In [17]:
cols = ["hour", "dow", "roll3_leaky"]
model = LinearRegression().fit(train[cols], train["y"])
print("leaky  R2 =", round(r2_score(test["y"], model.predict(test[cols])), 4),
      "| coefficient on roll3_leaky =", round(model.coef_[2], 3))

leaky  R2 = 0.8819 | coefficient on roll3_leaky = 1.053


In [18]:
cols = ["hour", "dow", "roll3_honest"]
model = LinearRegression().fit(train[cols], train["y"])
print("honest R2 =", round(r2_score(test["y"], model.predict(test[cols])), 4),
      "| coefficient on roll3_honest =", round(model.coef_[2], 3))

honest R2 = 0.5897 | coefficient on roll3_honest = 0.839


Same information, but the leaky version scores far higher (about 0.88 vs 0.59) because a third of the feature *is* the target.
With a 24-hour window the target is only 1/24 of the feature, so the gap is small — that is exactly why it survives code review.

**Interview check:** "Is `shift(1)` necessary here?" → answer with the decision time, not with a rule of thumb.
If the target is y[t+24] and y[t] is known at t, a window that ends at t is fine.

## 4. `diff` and `pct_change`

`diff()` is `y − y.shift(1)`; `pct_change()` is `y / y.shift(1) − 1`. Both look back, so they are fine at t.

In [19]:
pd.DataFrame({"y": y, "diff": y.diff(), "pct_change": y.pct_change()})

,y,diff,pct_change
2023-01-01 00:00:00,10,NaN,NaN
2023-01-01 01:00:00,20,10.0,1.000000
2023-01-01 02:00:00,30,10.0,0.500000
2023-01-01 03:00:00,40,10.0,0.333333
2023-01-01 04:00:00,50,10.0,0.250000
2023-01-01 05:00:00,60,10.0,0.200000


At 01:00: diff = 20 − 10 = 10; pct_change = 20/10 − 1 = 1.0 (a 100 % rise).

**Pitfall:** `pct_change` explodes near zero and makes no sense across a sign change. Prices can be negative.

In [20]:
price = pd.Series([10.0, 0.5, -5.0, 5.0])
pd.DataFrame({"price": price, "diff": price.diff(), "pct_change": price.pct_change()})

,price,diff,pct_change
0,10.0,NaN,NaN
1,0.5,-9.5,-0.95
2,-5.0,-5.5,-11.00
3,5.0,10.0,-2.00


0.5 → −5 shows as −1100 %; −5 → 5 shows as −200 % although the price went **up**. Use `diff` for prices.

**Pitfall:** old pandas silently forward-filled NaN inside `pct_change`. Pass `fill_method=None` so a gap stays a gap.

In [21]:
s = pd.Series([100.0, np.nan, 110.0])
pd.DataFrame({"s": s, "pct_change(fill_method=None)": s.pct_change(fill_method=None)})

,s,pct_change(fill_method=None)
0,100.0,NaN
1,NaN,NaN
2,110.0,NaN


## 5. Calendar features: numeric, one-hot, sin/cos

As a plain number, hour 23 is "far" from hour 0 although they are adjacent. Two fixes:
one-hot (24 columns, no order) or sin/cos (2 columns, keeps adjacency).

In [22]:
hours = pd.Series([0, 6, 12, 18], name="hour")
pd.DataFrame({
    "hour": hours,
    "sin": np.sin(2 * np.pi * hours / 24).round(3),
    "cos": np.cos(2 * np.pi * hours / 24).round(3),
})

,hour,sin,cos
0,0,0.0,1.0
1,6,1.0,0.0
2,12,0.0,-1.0
3,18,-1.0,-0.0


The (sin, cos) pair walks around a circle: hour 0 and hour 24 land on the same point, hour 6 and hour 18 are opposite.

In [23]:
pd.get_dummies(hours, prefix="h", dtype=int)

,h_0,h_6,h_12,h_18
0,1,0,0,0
1,0,1,0,0
2,0,0,1,0
3,0,0,0,1


One-hot: one column per hour value present. **Pitfall:** built separately on train and test, the two can have different columns if an hour is missing in one of them. Build on the full column, or `reindex(columns=...)`.

On the real data, same target (h=24) and lags, three encodings of hour:

In [24]:
hour = df["time"].dt.hour
base = pd.DataFrame({
    "lag24": cons.shift(24),
    "lag48": cons.shift(48),
    "is_weekend": (df["time"].dt.dayofweek >= 5).astype(int),
    "y": cons.shift(-24),
})
enc_numeric = pd.DataFrame({"hour": hour})
enc_sincos = pd.DataFrame({"hour_sin": np.sin(2 * np.pi * hour / 24), "hour_cos": np.cos(2 * np.pi * hour / 24)})
enc_onehot = pd.get_dummies(hour, prefix="h", dtype=float)

In [25]:
for name, enc in [("numeric", enc_numeric), ("sin/cos", enc_sincos), ("one-hot", enc_onehot)]:
    fr = pd.concat([base, enc], axis=1).dropna()
    tr, te = fr.iloc[:int(len(fr) * 0.8)], fr.iloc[int(len(fr) * 0.8):]
    Xc = [c for c in fr.columns if c != "y"]
    m = LinearRegression().fit(tr[Xc], tr["y"])
    print(f"{name:8s} test R2 = {r2_score(te['y'], m.predict(te[Xc])):.4f}")

numeric  test R2 = 0.8062
sin/cos  test R2 = 0.8107
one-hot  test R2 = 0.8296


## 6. Exogenous variables: what did you know at decision time?

Using the **measured** temperature at t + 24 to predict consumption at t + 24 is cheating: at
decision time t that number did not exist. What existed was a **forecast** issued at some
`origin` ≤ t.

Toy: two forecasts for the same three target hours, one issued at 00:00 and one at 12:00.
The decision is made at **06:00**.

In [26]:
fc_toy = pd.DataFrame({
    "origin":      pd.to_datetime(["2023-01-01 00:00"] * 3 + ["2023-01-01 12:00"] * 3),
    "target_time": pd.to_datetime(["2023-01-01 13:00", "2023-01-01 14:00", "2023-01-01 15:00"] * 2),
    "temp_fc":     [5.0, 6.0, 7.0,   5.5, 6.5, 7.5],
})
fc_toy

,origin,target_time,temp_fc
0,2023-01-01 00:00:00,2023-01-01 13:00:00,5.0
1,2023-01-01 00:00:00,2023-01-01 14:00:00,6.0
2,2023-01-01 00:00:00,2023-01-01 15:00:00,7.0
3,2023-01-01 12:00:00,2023-01-01 13:00:00,5.5
4,2023-01-01 12:00:00,2023-01-01 14:00:00,6.5
5,2023-01-01 12:00:00,2023-01-01 15:00:00,7.5


In [27]:
decisions = pd.DataFrame({
    "decision_time": pd.to_datetime(["2023-01-01 06:00"] * 3),
    "target_time":   pd.to_datetime(["2023-01-01 13:00", "2023-01-01 14:00", "2023-01-01 15:00"]),
})
decisions

,decision_time,target_time
0,2023-01-01 06:00:00,2023-01-01 13:00:00
1,2023-01-01 06:00:00,2023-01-01 14:00:00
2,2023-01-01 06:00:00,2023-01-01 15:00:00


**Right join:** for each target hour (`by="target_time"`), take the forecast with the latest
`origin` that is ≤ the decision time (`direction="backward"`). Both sides sorted on the `on` keys.

In [28]:
right = pd.merge_asof(
    decisions.sort_values("decision_time"),
    fc_toy.sort_values("origin"),
    left_on="decision_time", right_on="origin",
    by="target_time", direction="backward",
)
right

,decision_time,target_time,origin,temp_fc
0,2023-01-01 06:00:00,2023-01-01 13:00:00,2023-01-01,5.0
1,2023-01-01 06:00:00,2023-01-01 14:00:00,2023-01-01,6.0
2,2023-01-01 06:00:00,2023-01-01 15:00:00,2023-01-01,7.0


Every row got the 00:00 forecast (5.0, 6.0, 7.0): the 12:00 forecast came after the 06:00 decision.

**Wrong join:** "take the most recent forecast for that hour". It picks the 12:00 origin, which did not exist at 06:00.

In [29]:
latest = fc_toy.sort_values("origin").groupby("target_time").last().reset_index()
wrong = decisions.merge(latest, on="target_time")
wrong

,decision_time,target_time,origin,temp_fc
0,2023-01-01 06:00:00,2023-01-01 13:00:00,2023-01-01 12:00:00,5.5
1,2023-01-01 06:00:00,2023-01-01 14:00:00,2023-01-01 12:00:00,6.5
2,2023-01-01 06:00:00,2023-01-01 15:00:00,2023-01-01 12:00:00,7.5


Same target hours, but the forecast values 5.5, 6.5, 7.5 come from an origin **after** the decision time.
A check that catches this every time: `(origin > decision_time).any()` must be False.

In [30]:
print("right join uses a future origin:", (right["origin"] > right["decision_time"]).any())
print("wrong join uses a future origin:", (wrong["origin"] > wrong["decision_time"]).any())

right join uses a future origin: False
wrong join uses a future origin: True


The real forecast file: issued every 00:00 and 12:00 UTC, horizons 1–48 h, so each target hour has up to four forecasts.

In [31]:
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
fc[fc["forecast_datetime"] == pd.Timestamp("2022-03-10 15:00", tz="UTC")]

,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
6470,2022-03-09 00:00:00+00:00,2022-03-10 15:00:00+00:00,39,6.51
6506,2022-03-09 12:00:00+00:00,2022-03-10 15:00:00+00:00,27,7.53
6542,2022-03-10 00:00:00+00:00,2022-03-10 15:00:00+00:00,15,7.99
6578,2022-03-10 12:00:00+00:00,2022-03-10 15:00:00+00:00,3,6.00


Build the real frame in decision-time coordinates (decision at t, target at t + 24) and join the honest forecast.

In [32]:
H = 24
frame = pd.DataFrame({
    "decision_time": df["time"],
    "target_time": df["time"] + pd.Timedelta(hours=H),
    "y": cons.shift(-H),
    "cons_now": cons,
    "cons_lag24": cons.shift(24),
    "temp_now": df["temp_c"],
    "temp_actual_target": df["temp_c"].shift(-H),     # measured at t+24: NOT known at t
})
frame.head(2)

,decision_time,target_time,y,cons_now,cons_lag24,temp_now,temp_actual_target
0,2022-01-01 00:00:00+00:00,2022-01-02 00:00:00+00:00,26785.3,26858.4,NaN,0.11,0.15
1,2022-01-01 01:00:00+00:00,2022-01-02 01:00:00+00:00,25826.5,26177.8,NaN,-0.18,-0.59


In [33]:
fc_r = fc.rename(columns={"forecast_datetime": "target_time"}).sort_values("origin_datetime")
honest = pd.merge_asof(
    frame.sort_values("decision_time"),
    fc_r[["origin_datetime", "target_time", "horizon_h", "temp_forecast_c"]],
    left_on="decision_time", right_on="origin_datetime",
    by="target_time", direction="backward",
)
honest[["decision_time", "target_time", "origin_datetime", "horizon_h", "temp_forecast_c", "temp_actual_target"]].iloc[100:103]

,decision_time,target_time,origin_datetime,horizon_h,temp_forecast_c,temp_actual_target
100,2022-01-05 04:00:00+00:00,2022-01-06 04:00:00+00:00,2022-01-05 00:00:00+00:00,28.0,-0.84,-2.85
101,2022-01-05 05:00:00+00:00,2022-01-06 05:00:00+00:00,2022-01-05 00:00:00+00:00,29.0,-3.35,-1.99
102,2022-01-05 06:00:00+00:00,2022-01-06 06:00:00+00:00,2022-01-05 00:00:00+00:00,30.0,-0.18,-1.25


In [34]:
print("uses a future origin :", (honest["origin_datetime"] > honest["decision_time"]).any())
print("horizons used        :", honest["horizon_h"].min(), "to", honest["horizon_h"].max(), "hours")
print("forecast error mean  :", round((honest["temp_forecast_c"] - honest["temp_actual_target"]).mean(), 2), "C")

uses a future origin : False
horizons used        : 24.0 to 35.0 hours
forecast error mean  : 0.3 C


Horizons 24–35 h: the forecast for hour t + 24 was issued 0–11 hours before the decision. Now the comparison that matters.

In [35]:
common = ["cons_now", "cons_lag24"]
fr = honest.dropna(subset=common + ["y", "temp_actual_target", "temp_forecast_c", "temp_now"])
tr, te = fr.iloc[:int(len(fr) * 0.8)], fr.iloc[int(len(fr) * 0.8):]
for name, extra in [("actual temp at t+24 (cheat)", ["temp_actual_target"]),
                    ("forecast known at t        ", ["temp_forecast_c"]),
                    ("temp at t (lagged)         ", ["temp_now"]),
                    ("no temperature             ", [])]:
    c = common + extra
    m = LinearRegression().fit(tr[c], tr["y"])
    print(name, " R2 =", round(r2_score(te["y"], m.predict(te[c])), 4))

actual temp at t+24 (cheat)  R2 = 0.847
forecast known at t          R2 = 0.8466
temp at t (lagged)           R2 = 0.8468
no temperature               R2 = 0.8465


The "cheat" number is what you would report by accident after merging actual weather on `time`.
The gap between cheat and forecast is the value of a perfect weather forecast, not model skill.

## 7. Scaler leakage

`StandardScaler` subtracts a mean and divides by a standard deviation. Fit on the full sample,
those numbers contain the test period.

Toy: five values; the last one belongs to the test set.

In [36]:
vals = pd.Series([1.0, 2.0, 3.0, 4.0, 100.0], index=["train", "train", "train", "train", "test"])
print("mean of train only :", vals.iloc[:4].mean())
print("mean of everything :", vals.mean())

mean of train only : 2.5
mean of everything : 22.0


Scaled with the full-sample mean (22), every training value becomes strongly negative — the
training data has been told that something big is coming. Fit on train only, or put the scaler
in a `Pipeline` so it can only ever see what `.fit()` sees.

In [37]:
from sklearn.preprocessing import StandardScaler
sc_train = StandardScaler().fit(vals.iloc[:4].to_frame())
sc_full = StandardScaler().fit(vals.to_frame())
pd.DataFrame({"value": vals, "scaled_with_train": sc_train.transform(vals.to_frame())[:, 0].round(2),
              "scaled_with_full": sc_full.transform(vals.to_frame())[:, 0].round(2)})

,value,scaled_with_train,scaled_with_full
train,1.0,-1.34,-0.54
train,2.0,-0.45,-0.51
train,3.0,0.45,-0.49
train,4.0,1.34,-0.46
test,100.0,87.21,2.00


## 8. Group means computed on the full sample

"Mean y for this group" is a fine feature if the mean comes from **train only**. Computed with
`groupby().transform("mean")` on the whole frame, each row's feature contains its own target.

Toy: four rows, two groups, the last two rows are test.

In [38]:
g = pd.DataFrame({"group": ["A", "B", "A", "B"], "y": [10, 100, 30, 300]}, index=["train", "train", "test", "test"])
g["mean_full_sample"] = g.groupby("group")["y"].transform("mean")
g

,group,y,mean_full_sample
train,A,10,20.0
train,B,100,200.0
test,A,30,20.0
test,B,300,200.0


The test row for A has feature 20 = (10 + 30) / 2: its own target (30) is inside the feature.

Correct: compute the means on the train rows and `map` them onto every row.

In [39]:
train_means = g.loc["train"].groupby("group")["y"].mean()
print(train_means)
g["mean_train_only"] = g["group"].map(train_means)
g

group
A     10.0
B    100.0
Name: y, dtype: float64


,group,y,mean_full_sample,mean_train_only
train,A,10,20.0,10.0
train,B,100,200.0,100.0
test,A,30,20.0,10.0
test,B,300,200.0,100.0


Now the A rows get 10 (the only A value in train). The test targets never entered the feature.

## 9. Overlapping labels and shuffled cross-validation

With h = 24 the targets of rows t and t+1 are consumption one hour apart — almost the same number.
A shuffled K-fold puts row t in test and rows t ± 1 in train, so a flexible model "predicts" the
test row by copying its neighbours.

Toy: consecutive targets are near-duplicates.

In [40]:
tgt24 = cons.shift(-24).dropna()
print("correlation of the h=24 target with itself one row later :", round(tgt24.autocorr(1), 3))

correlation of the h=24 target with itself one row later : 0.935


In [41]:
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.tree import DecisionTreeRegressor

fr = pd.DataFrame({"cons_now": cons, "cons_lag24": cons.shift(24), "hour": hour, "y": cons.shift(-24)}).dropna()
Xn, yn = fr[["cons_now", "cons_lag24", "hour"]], fr["y"]
tree = DecisionTreeRegressor(random_state=0)     # deliberately over-flexible
print("KFold shuffled  R2 :", round(cross_val_score(tree, Xn, yn, cv=KFold(5, shuffle=True, random_state=0)).mean(), 3))
print("TimeSeriesSplit R2 :", round(cross_val_score(tree, Xn, yn, cv=TimeSeriesSplit(5)).mean(), 3))

KFold shuffled  R2 : 0.715


TimeSeriesSplit R2 : 0.626


The shuffled score is higher and is not achievable in production.

**Interview check:** "Why shouldn't we shuffle?" → autocorrelated labels plus neighbouring rows in train.
Also add a **gap** of h rows between train and test (`TimeSeriesSplit(gap=24)`), otherwise the last
training targets are future values relative to the first test decision.

## 10. Filling gaps: `ffill` is stale, `bfill` and `interpolate` are from the future

In [42]:
s = pd.Series([10.0, np.nan, np.nan, 40.0], index=pd.date_range("2023-01-01", periods=4, freq="h"))
pd.DataFrame({"raw": s, "ffill": s.ffill(), "bfill": s.bfill(), "interpolate": s.interpolate()})

,raw,ffill,bfill,interpolate
2023-01-01 00:00:00,10.0,10.0,10.0,10.0
2023-01-01 01:00:00,NaN,10.0,40.0,20.0
2023-01-01 02:00:00,NaN,10.0,40.0,30.0
2023-01-01 03:00:00,40.0,40.0,40.0,40.0


At 01:00: `ffill` repeats 10 (old but known), `bfill` gives 40 (the value at 03:00 — the future),
`interpolate` gives 20 (needs the 03:00 value too). As a feature "as of 01:00", only `ffill` is honest.

## 11. `resample`: `label` and `closed`

Four half-hours with values 1, 2, 3, 4. Summed to hourly bins two ways.

In [43]:
s = pd.Series([1, 2, 3, 4], index=pd.date_range("2023-01-01 00:00", periods=4, freq="30min"))
s

2023-01-01 00:00:00    1
2023-01-01 00:30:00    2
2023-01-01 01:00:00    3
2023-01-01 01:30:00    4
Freq: 30min, dtype: int64

In [44]:
s.resample("h").sum()                                   # default: label=left, closed=left

2023-01-01 00:00:00    3
2023-01-01 01:00:00    7
Freq: h, dtype: int64

Default: the bin labelled 00:00 holds 00:00 + 00:30 = 3; the bin labelled 01:00 holds 01:00 + 01:30 = 7.
The 00:00 bin is only complete at 01:00, although its label says 00:00.

In [45]:
s.resample("h", label="right", closed="right").sum()

2023-01-01 00:00:00    1
2023-01-01 01:00:00    5
2023-01-01 02:00:00    4
Freq: h, dtype: int64

With `closed="right"` a bin runs from just after one hour up to and including the next: the bin
labelled 01:00 holds 00:30 + 01:00 = 5, the one labelled 02:00 holds 01:30 only = 4, and the
00:00 label holds just the 00:00 value = 1. Same data, different numbers per label.

Either way a daily mean labelled "2022-01-05" is only known at the **next** midnight. Make that
explicit before joining it back to hourly rows.

In [46]:
hourly = df.set_index("time")["consumption_mwh"]
daily = hourly.resample("D").mean().rename("prev_day_mean").reset_index()
daily["available_from"] = daily["time"] + pd.Timedelta(days=1)
daily.head(3)

,time,prev_day_mean,available_from
0,2022-01-01 00:00:00+00:00,30587.241667,2022-01-02 00:00:00+00:00
1,2022-01-02 00:00:00+00:00,30374.750000,2022-01-03 00:00:00+00:00
2,2022-01-03 00:00:00+00:00,31855.712500,2022-01-04 00:00:00+00:00


In [47]:
joined = pd.merge_asof(hourly.reset_index(), daily[["available_from", "prev_day_mean"]],
                       left_on="time", right_on="available_from", direction="backward")
joined.set_index("time").loc["2022-01-05 22:00":"2022-01-06 01:00"]

,consumption_mwh,available_from,prev_day_mean
time,,,
2022-01-05 22:00:00+00:00,32604.5,2022-01-05 00:00:00+00:00,32808.137500
2022-01-05 23:00:00+00:00,30608.3,2022-01-05 00:00:00+00:00,32808.137500
2022-01-06 00:00:00+00:00,30808.0,2022-01-06 00:00:00+00:00,33942.295833
2022-01-06 01:00:00+00:00,30059.8,2022-01-06 00:00:00+00:00,33942.295833


The 2022-01-05 daily mean first appears on rows from 2022-01-06 00:00 onwards.

## 12. Leakage checklist

1. What is the **decision time** of each row, and what is the **horizon**?
2. Is the frame **sorted**, with **unique timestamps** and a **constant interval**? (Otherwise `shift(k)` is not k hours.)
3. Does every feature use only data with timestamp **≤ decision time**? Look at every `rolling`, `expanding`, `diff`, `pct_change` for a missing `shift`.
4. Any `center=True`, `bfill`, `interpolate`, `shift(-k)` that is not the target?
5. Were exogenous variables **measured** at the target time or **forecast** at the decision time? Which vintage?
6. Were scalers / imputers / encoders fit on **train only** (ideally inside a `Pipeline`)?
7. Are group statistics computed on **train only**?
8. Was the split **chronological**, with a **gap ≥ h**?
9. Cross-validation: `TimeSeriesSplit`, never shuffled folds, when labels overlap.
10. Is the **target** what you think (units, sign, horizon)? Check one row by hand.
11. After merges: did the row count change? Any right-table timestamps **after** the decision time?
12. Is the result **too good**? An R² that jumps after a small change is a bug until proven otherwise.